# 05 — Evaluation & Explainability

Walk-forward CV, coverage analysis, and SHAP feature importance.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
from pathlib import Path

ROOT = Path('..')

In [ ]:
from src.data.fetch_data import fetch_entsoe_load, fetch_openmeteo_weather
from src.data.preprocess import build_features, train_test_split_temporal

load    = fetch_entsoe_load()
weather = fetch_openmeteo_weather()
df      = build_features(load, weather)
train, test = train_test_split_temporal(df)

## 1. Walk-forward cross-validation

In [ ]:
from src.models.ml_models import LightGBMForecaster
from src.evaluation.metrics import walk_forward_cv

cv_results = walk_forward_cv(
    df,
    model_factory=LightGBMForecaster,
    n_folds=5,
    horizon=168,
)
print(cv_results)
print('\nMean metrics:')
print(cv_results.drop(columns='fold').mean())

## 2. Coverage analysis

In [ ]:
from src.models.ml_models import LightGBMForecaster
from src.evaluation.metrics import evaluate_forecasts

lgbm = LightGBMForecaster()
lgbm.fit(train)

X_test = test.drop(columns=['load_MW'])
preds  = lgbm.predict(X_test)

metrics = evaluate_forecasts(
    test['load_MW'], preds['forecast'],
    lower_80=preds['lower_80'], upper_80=preds['upper_80'],
)
print(metrics)

## 3. SHAP analysis

In [ ]:
from src.evaluation.explainability import compute_shap_tree, plot_summary, top_features

shap_values = compute_shap_tree(lgbm.models[0.5], X_test, model_name='lightgbm')

print('Top 10 features by mean |SHAP|:')
print(top_features(shap_values))

In [ ]:
plot_summary(shap_values, model_name='lightgbm')
shap.summary_plot(shap_values, X_test.sample(500, random_state=42))

## 4. Waterfall — single prediction explanation

In [ ]:
from src.evaluation.explainability import plot_waterfall

plot_waterfall(shap_values, sample_idx=0, model_name='lightgbm')
shap.plots.waterfall(shap_values[0])

## 5. Residual analysis

In [ ]:
from src.visualization.plots import plot_residuals

plot_residuals(test['load_MW'], preds['forecast'], model_name='LightGBM')
plt.show()